# `dsfb-rf` Colab Reproducibility Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infinityabundance/dsfb/blob/main/crates/dsfb-rf/colab/dsfb_rf_reproduce.ipynb)

**Companion paper:** `dsfb_rf_v2.tex` &mdash; *DSFB Structural Semiotics Engine for RF Signal Monitoring* (v2.0, April 2026).
**Crate:** [crates.io/crates/dsfb-rf](https://crates.io/crates/dsfb-rf) &mdash; Apache-2.0.
**License of this notebook:** Apache-2.0.

---

## 1. What this notebook does, in one paragraph

It builds the `dsfb-rf` Rust crate **from source on a fresh Colab runtime**, prepares an eight-dataset slice catalog, runs `generate_figures_all` to emit the paper's 67 figures plus a merged PDF and bit-exact JSON, bundles everything (including the slices and a provenance manifest) into a single artefact zip, and prompts your browser to download it. Expected wall-clock on a free-tier CPU runtime: about 15 minutes.

## 2. What DSFB is

DSFB &mdash; *Drift-Slew Fusion Bootstrap* &mdash; observes the **residual stream** $r(k)$ that any supervised receiver already produces (demodulator error, track innovation, Kalman residual, compressed-sensing residual). It treats that stream as a typed trajectory on the semiotic manifold

$$\mathcal{M} = \{(\|r\|,\; \dot{r},\; \ddot{r})\}$$

and classifies the discrete sign tuple

$$\sigma(k) = \big(\mathrm{sign}(\|r\|),\; \mathrm{sign}(\dot{r}),\; \mathrm{sign}(\ddot{r})\big) \in \{-1, 0, +1\}^3.$$

A *Grammar FSM* over $\sigma(k)$ emits the episodes the paper evaluates. The envelope exit time is finite-time-bounded:

$$t^{*} \;\le\; \rho / \alpha \qquad \text{(Theorem 1)}$$

where $\rho$ is calibrated from a healthy window and $\alpha$ is the observed drift. The crate is `no_std`, `no_alloc`, `#![forbid(unsafe_code)]`, and ships Kani proofs for grammar panic-freedom, envelope consistency, and Q16.16 round-trip safety.

**Positioning.** DSFB does *not* classify emitters, replace a detector, or guarantee hard real-time bounds. It is an observer layer that reads what the receiver already computes and asks — structurally — whether the residual trajectory is *organised* or *noise*.

## 3. Why a Colab notebook

Reviewers and SBIR program managers should not need a local Rust toolchain to verify the figure-generation pipeline. A one-click path from the paper to a running crate closes the reviewer-confidence gap. The notebook is *orchestration only* &mdash; it builds the published v1.0.0 crate verbatim and never mutates `src/`, `Cargo.toml`, or the paper.

## 4. What the notebook does NOT do (academic honesty)

- **It does not render the paper.** LaTeX is not installed in the notebook; the compiled PDF stays in the authors' local tree (gitignored).
- **It does not run `paper-lock`.** That binary needs the full 20 GB `GOLD_XYZ_OSC.0001_1024.hdf5` file, which does not fit on a free-tier Colab disk.
- **It does not reproduce Table 1 numerically.** Those headline precision/recall figures come from `paper-lock` on the full dataset; the notebook reproduces the *figures* from the crate's frozen synthetic signal models, with eight real-data slices shown alongside for contextual breadth.
- **It does not claim to reproduce any specific ORACLE, POWDER, Tampere, DeepBeam, DeepSense, or ColO-RAN benchmark.** Those datasets are shown as slice exhibits &mdash; a reviewer spot-check that the residual-trace shapes the figures depend on are consistent with what falls out of eight distinct real-world sources.
- **If a public mirror is unreachable**, the notebook falls back to a **loudly-labelled synthetic proxy** for that slot. Every proxy prints `[SYNTHETIC PROXY]` banners, stamps its files with `dsfb_rf:provenance="synthetic-proxy"`, and records the proxy origin in `SLICE_MANIFEST.json`. A proxy is *never* substituted for a paper metric.

## 5. How to use it

**Runtime &rarr; Run all.** Wait ~15 min (most of that is `cargo build --release`). A browser download prompt for `dsfb-rf-<timestamp>-artifacts.zip` appears at the end. Unzip it locally to inspect the 67 PDFs, the merged `dsfb-rf-all-figures.pdf`, `figure_data.json`, `figure_data_all.json`, the eight slice files, and `ARTIFACT_MANIFEST.json`.

## 6. License + citation

Cite as: *de Beer, R. "DSFB Structural Semiotics Engine for RF Signal Monitoring" (2026), v1.0.0, companion paper `dsfb_rf_v2.tex`.* Apache-2.0 for the crate and this notebook. RadioML 2018.01a slice: CC BY-NC-SA 4.0 derivative. Tampere GNSS slice: CC BY 4.0 derivative.

---


In [ ]:
# Cell 2 — runtime probe. Informational; no paper claim depends on this.
import os, platform, shutil, sys
print(f"python         : {sys.version.split()[0]}")
print(f"platform       : {platform.platform()}")
print(f"machine        : {platform.machine()}")
print(f"cpu_count      : {os.cpu_count()}")
try:
    total, used, free = shutil.disk_usage('/content')
    print(f"disk /content  : {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")
except Exception as e:
    print(f"disk /content  : unavailable ({e})")
try:
    import subprocess
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=4)
    print(f"nvidia-smi -L  : {(out.stdout.strip() or '(no GPU detected)') if out.returncode == 0 else '(not available)'}")
except Exception:
    print("nvidia-smi -L  : (not available)")

In [ ]:
# Cell 3 — system packages. libhdf5-dev is needed for hdf5-metno; poppler-utils + pdfunite merge PDFs.
# Probe-and-fallback pin: try reproducible versions first, fall back to the floating package. The resolved
# version is printed via `dpkg -l` so ARTIFACT_MANIFEST.json captures the exact package on this run.
!apt-get update -qq
!bash -c 'HDF5_CANDIDATES="libhdf5-dev=1.10.7+repack-4ubuntu2 libhdf5-dev=1.10.7+repack-4build1 libhdf5-dev"; for cand in $HDF5_CANDIDATES; do if apt-get install -y -qq $cand pkg-config poppler-utils > /dev/null 2>&1; then echo "[libhdf5] installed: $cand"; break; fi; done; dpkg -l libhdf5-dev 2>/dev/null | tail -1'

In [ ]:
# Cell 4 — Python deps. Pinned to the versions the crate's scripts are tested against.
!pip install -q 'h5py==3.16.*' 'numpy<2.3' 'matplotlib>=3.7' 'requests>=2.31'

In [ ]:
# Cell 5 — Rust 1.83.0 via rustup (matches rust-toolchain.toml).
import os, subprocess
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.83.0 --profile minimal -q
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + os.pathsep + os.environ['PATH']
print(subprocess.run(['rustc', '--version'], capture_output=True, text=True).stdout.strip())
print(subprocess.run(['cargo', '--version'], capture_output=True, text=True).stdout.strip())

In [ ]:
# Cell 6 — clone the dsfb repo (shallow, branch=main) and cd into the crate.
import os, subprocess
ROOT = '/content/dsfb'
CRATE = os.path.join(ROOT, 'crates', 'dsfb-rf')
if not os.path.exists(ROOT):
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'main',
                           'https://github.com/infinityabundance/dsfb.git', ROOT])
os.chdir(CRATE)
sha = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print(f'cwd    : {os.getcwd()}')
print(f'commit : {sha}')

## Dataset slices

The next cell prepares an **eight-dataset slice catalog** via `scripts/prepare_slices.py`. Each slice is capped at ~2 MB so the bundled zip stays small.

| # | Slice | Real source | Provenance posture in Colab |
|---|---|---|---|
| 1 | RadioML 2018.01a | DeepSig | **real**, shipped in repo (`data/slices/radioml_2018_slice.hdf5`, 1.85 MB slice of `GOLD_XYZ_OSC.0001_1024.hdf5`, CC BY-NC-SA 4.0 derivative). |
| 2 | ORACLE Raw IQ | Northeastern GENESYS, USRP X310 | **real** head-slice shipped in repo (extracted from author's 28 GB zip; schema-identical SigMF pair). |
| 3 | POWDER | Genesys Lab, PAWR | **real** head-slice shipped in repo (extracted from author's 4.3 GB zip; schema-identical `.bin` + `.json`). |
| 4 | Tampere GNSS | Wang/Sankari/Lohan/Valkama (Zenodo 10.5281/zenodo.13846381, CC BY 4.0) | attempts live download of the first 2 MB of `Data.zip`; proxy fallback on timeout. |
| 5 | ColO-RAN | `wineslab/colosseum-oran-coloran-dataset` | attempts live download from GitHub raw; proxy fallback. **Non-IQ companion annex** (DU KPI CSVs). |
| 6 | ColO-RAN-commag | `wineslab/colosseum-oran-commag-dataset` | attempts live download from GitHub raw; proxy fallback. **Non-IQ companion annex** (scheduling KPIs). |
| 7 | DeepBeam | Northeastern repository (`neu:ww72bh952`) | **real** head-slice of user-downloaded `neu_ww72bk394.h5` (59 GB parent, 11.06 B IQ rows); native NI schema `/iq (N,2)`, `/gain`, `/rx_beam`, `/tx_beam`. |
| 8 | DeepSense-6G Scenario 23 (UAV mmWave) | `deepsense6g.net/scenarios/scenario-23` | **real** head-slice extracted from user-downloaded `scenario23_dev_w_resources.zip` (HDF5: mmWave power N×64 + UAV telemetry). |

**Reminder.** Proxies are clearly labelled and never substitute for paper metrics; they are contextual exhibits. No figure in the merged PDF reads any slice file &mdash; the figures come from the crate's deterministic synthetic models, frozen at v1.0.0.


In [ ]:
# Cell 8 — prepare eight-dataset slice catalog.
import subprocess
rc = subprocess.call(['python3', 'scripts/prepare_slices.py'])
if rc != 0:
    raise SystemExit(f'prepare_slices.py failed with exit code {rc}')

In [ ]:
# Cell 9 — per-slice sanity summary. Pure print; no figure metric depends on this.
import json, os
with open('data/slices/SLICE_MANIFEST.json') as fh:
    manifest = json.load(fh)
slices = manifest.get('slices', [])
print(f"{'name':<18} {'provenance':<18} {'bytes':>10}  {'sha256 (head)':<18} files")
print('-' * 95)
for s in slices:
    print(f"{s['name']:<18} {s['provenance']:<18} {s['bytes']:>10}  {s['sha256'][:16]:<18} {', '.join(s['files'])}")
print()
print(f"manifest schema_version : {manifest.get('schema_version')}")
print(f"generated_at            : {manifest.get('generated_at')}")

In [ ]:
# Cell 10 — the canonical end-to-end invocation. Compiles the crate in release and
# emits 67 figures + merged PDF + JSON + zip into ../dsfb-rf-output/.
# Expected wall-clock on a free-tier CPU runtime: ~10–14 min.
!cargo run --release --example generate_figures_all --features std,serde

In [ ]:
# Cell 10b — additive Real-Dataset Figure Bank (80 figures, 10 per slice × 8 slices).
# DSFB is framed as the structural interpreter of each upstream producer's
# residual (matched filter / AGC / channel estimator / GNSS tracking loop /
# scheduler EWMA / beamformer / beam-tracker). Not a detector, not a replacement.
import subprocess
rc = subprocess.call([
    'cargo', 'run', '--release',
    '--example', 'generate_figures_real',
    '--features', 'std,serde,real_figures',
])
print('exit code:', rc)
assert rc == 0, 'generate_figures_real failed'


In [ ]:
# Cell 11 — verify the figure output across both synthetic and real-world banks.
import glob, os

# Synthetic bank (67 figures from generate_figures_all).
runs = sorted(glob.glob('../dsfb-rf-output/dsfb-rf-*/'))
# Exclude the dsfb-rf-real-* runs from the synthetic roster.
runs_synth = [r for r in runs if '/dsfb-rf-real-' not in r]
if not runs_synth:
    raise SystemExit('No synthetic dsfb-rf-output/dsfb-rf-* folder found.')
latest = runs_synth[-1]
pdfs = glob.glob(os.path.join(latest, 'figs', '*.pdf'))
merged = os.path.join(latest, 'dsfb-rf-all-figures.pdf')
jsons = [os.path.join(latest, 'figure_data.json'), os.path.join(latest, 'figure_data_all.json')]
zips = glob.glob(os.path.join(latest, 'dsfb-rf-*-artifacts.zip'))
print(f'latest synth run   : {latest}')
print(f'synthetic PDFs     : {len(pdfs)} (expect >= 67)')
print(f'merged PDF         : {os.path.exists(merged)}')
for j in jsons:
    print(f'  {os.path.basename(j):<22}: {os.path.exists(j)}')
print(f'artefact zip       : {zips[0] if zips else "MISSING"}')
assert len(pdfs) >= 67, f'expected >= 67 synthetic PDFs, got {len(pdfs)}'
assert os.path.exists(merged), 'merged synthetic PDF missing'
assert all(os.path.exists(j) for j in jsons), 'figure_data*.json missing'
assert zips, 'artefact zip missing'

# Real-world bank (80 figures + fig_149 demodulation-threshold scan).
runs_real = sorted(glob.glob('../dsfb-rf-output/dsfb-rf-real-*/'))
latest_real = runs_real[-1] if runs_real else None
assert latest_real, 'No dsfb-rf-output/dsfb-rf-real-* folder found.'
pdfs_real = glob.glob(os.path.join(latest_real, 'figs', '*.pdf'))
merged_real = os.path.join(latest_real, 'dsfb-rf-all-real-figures.pdf')
json_real = os.path.join(latest_real, 'figure_data_real.json')
print(f'latest real run    : {latest_real}')
print(f'real-world PDFs    : {len(pdfs_real)} (expect >= 80, +1 for fig_149)')
print(f'merged real PDF    : {os.path.exists(merged_real)}')
print(f'figure_data_real   : {os.path.exists(json_real)}')
assert len(pdfs_real) >= 80, f'expected >= 80 real-world PDFs, got {len(pdfs_real)}'
assert os.path.exists(merged_real), 'merged real-world PDF missing'
assert os.path.exists(json_real), 'figure_data_real.json missing'

# fig_149 is only produced when the RadioML slice is real; it is optional so the
# assertion is conditional on the fig_149 data block being present in figure_data_real.json.
import json as _json
with open(json_real) as _fh:
    _data_real = _json.load(_fh)
_radioml = _data_real.get('radioml') or {}
if _radioml.get('fig_149'):
    fig149_pdfs = glob.glob(os.path.join(latest_real, 'figs', 'fig_149_*.pdf'))
    assert fig149_pdfs, 'fig_149 data present but no fig_149_*.pdf rendered'
    print(f'fig_149 (demod thr): present ✓  ({os.path.basename(fig149_pdfs[0])})')
else:
    print('fig_149 (demod thr): skipped (RadioML slice missing or block skipped)')

total = len(pdfs) + len(pdfs_real)
print(f'\ntotal figures      : {total} (≈ 67 + 80 + 1)')
print('all checks PASS')

In [ ]:
# Cell 12 — add slice files + real-world figure bank + ARTIFACT_MANIFEST.json to the existing zip.
import glob, json, os, subprocess, time, zipfile
runs = sorted(glob.glob('../dsfb-rf-output/dsfb-rf-*/'))
runs_synth = [r for r in runs if '/dsfb-rf-real-' not in r]
runs_real = sorted(glob.glob('../dsfb-rf-output/dsfb-rf-real-*/'))
latest = runs_synth[-1]
latest_real = runs_real[-1] if runs_real else None
zip_path = glob.glob(os.path.join(latest, 'dsfb-rf-*-artifacts.zip'))[0]
slice_files = sorted(glob.glob('data/slices/*'))
commit = subprocess.run(['git', '-C', '/content/dsfb', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
cargo_ver = subprocess.run(['cargo', '--version'], capture_output=True, text=True).stdout.strip()
libhdf5_pkg = subprocess.run(['bash', '-c', "dpkg -l libhdf5-dev 2>/dev/null | tail -1 | tr -s ' '"], capture_output=True, text=True).stdout.strip()
with open('data/slices/SLICE_MANIFEST.json') as fh:
    slice_manifest = json.load(fh)
artefact_manifest = {
    'schema_version': 2,
    'notebook': 'crates/dsfb-rf/colab/dsfb_rf_reproduce.ipynb',
    'generated_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'git_commit': commit,
    'cargo_version': cargo_ver,
    'libhdf5_dev': libhdf5_pkg,
    'synth_run_dir': os.path.relpath(latest),
    'real_run_dir': os.path.relpath(latest_real) if latest_real else None,
    'slices': [{k: v for k, v in s.items() if k != 'reason'} for s in slice_manifest.get('slices', [])],
    'notice': (
        'This zip bundles the 67 synthetic figure PDFs (generate_figures_all), '
        'the 80+1 real-world figure PDFs (generate_figures_real), both merged PDFs, '
        'figure_data JSONs, eight real-dataset slice exhibits, and this manifest. '
        'fig_149 is the additive demodulation-threshold scan over 24 RadioML modulations; '
        'produced only when the RadioML slice is real. '
        'Proxy slices are marked provenance="synthetic-proxy" and are never substituted for paper metrics.'
    ),
}
am_path = os.path.join(latest, 'ARTIFACT_MANIFEST.json')
with open(am_path, 'w') as fh:
    json.dump(artefact_manifest, fh, indent=2)
with zipfile.ZipFile(zip_path, 'a', compression=zipfile.ZIP_DEFLATED) as zf:
    existing = set(zf.namelist())
    if 'ARTIFACT_MANIFEST.json' not in existing:
        zf.write(am_path, 'ARTIFACT_MANIFEST.json')
    if 'SLICE_MANIFEST.json' not in existing:
        zf.write('data/slices/SLICE_MANIFEST.json', 'slices/SLICE_MANIFEST.json')
    for sf in slice_files:
        arc = 'slices/' + os.path.basename(sf)
        if arc not in existing:
            zf.write(sf, arc)
    # Real-world figure bank: add under a distinct subtree so synthetic consumers stay stable.
    if latest_real:
        real_pdfs = sorted(glob.glob(os.path.join(latest_real, 'figs', '*.pdf')))
        for rp in real_pdfs:
            arc = 'real-figures/figs/' + os.path.basename(rp)
            if arc not in existing:
                zf.write(rp, arc)
        merged_real = os.path.join(latest_real, 'dsfb-rf-all-real-figures.pdf')
        if os.path.exists(merged_real) and 'real-figures/dsfb-rf-all-real-figures.pdf' not in existing:
            zf.write(merged_real, 'real-figures/dsfb-rf-all-real-figures.pdf')
        fd_real = os.path.join(latest_real, 'figure_data_real.json')
        if os.path.exists(fd_real) and 'real-figures/figure_data_real.json' not in existing:
            zf.write(fd_real, 'real-figures/figure_data_real.json')
real_count = len(glob.glob(os.path.join(latest_real, 'figs', '*.pdf'))) if latest_real else 0
print(f'appended {len(slice_files) + 2 + real_count} files to {zip_path}')
print(f'final zip size: {os.path.getsize(zip_path)/1e6:.2f} MB')

In [ ]:
# Cell 13 — browser download prompt.
import glob, os
zip_path = glob.glob('../dsfb-rf-output/dsfb-rf-*/dsfb-rf-*-artifacts.zip')[-1]
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f'(not running in Colab) artefact zip at: {os.path.abspath(zip_path)}')

## Verification checklist (for a sceptical reviewer)

Unzip the downloaded bundle locally and confirm:

1. **67 figure PDFs** under `figs/` (synthetic bank from `generate_figures_all`).
2. **80 figure PDFs** under `real-figures/figs/` (real-world bank from `generate_figures_real`, 10 per slice × 8 slices).
3. **`fig_149_*_demod_threshold_scan.pdf`** — additive demodulation-threshold scan across 24 RadioML modulations; renders when the RadioML slice is real.
4. **Merged PDFs** `dsfb-rf-all-figures.pdf` and `real-figures/dsfb-rf-all-real-figures.pdf` both open cleanly.
5. **`figure_data.json` + `figure_data_all.json` + `real-figures/figure_data_real.json`** contain the bit-exact engine data that the Python renderers consumed.
6. **`slices/SLICE_MANIFEST.json`** lists eight slices; every row that says `provenance="synthetic-proxy"` is a stand-in for a mirror that was unreachable at run time, not a paper metric.
7. **`ARTIFACT_MANIFEST.json`** records the git commit SHA, cargo version, and resolved `libhdf5-dev` package version that produced this run &mdash; pin it in any follow-up discussion.

## What this notebook is *not* a substitute for

- Running `paper-lock` on the full 20 GB RadioML GOLD file to regenerate Table 1 numerics. See `REPRODUCE.md` §2.2.
- Running the Kani harnesses. Kani needs a CBMC toolchain; see `REPRODUCE.md` §5.
- Cross-target timing (Cortex-M4F, RISC-V 32-bit, x86-64). See `.github/workflows/qemu_timing.yml`.

## Troubleshooting

- **`cargo build` hangs on `hdf5-metno`**: ensure Cell 3 (`libhdf5-dev`) completed successfully.
- **Fewer than 67 PDFs**: check `dsfb-rf-output/dsfb-rf-*/stderr.log` (if present) or re-run Cell 10. The renderer spawns a Python subprocess; a missing `matplotlib` will leave PDFs missing.
- **Fewer than 80 real-world PDFs**: some slices intentionally skip on missing data; the `[SKIPPED — … not present]` banner from Cell 10b shows which. fig_149 requires the RadioML HDF5 slice to be real.
- **`[SYNTHETIC PROXY]` banners in Cell 8**: expected for most of slices 4&ndash;8 in a vanilla Colab session, because the corresponding public mirrors require authentication or are too large to stream. Slices 1&ndash;3 should always be real.

## Report issues

File at <https://github.com/infinityabundance/dsfb/issues>. Include `ARTIFACT_MANIFEST.json` and the `SLICE_MANIFEST.json` from your run so provenance is pinned.